# Lab 2: Monte Carlo Tree Search
In this notebook, we implement Monte Carlo Tree Search (MCTS) for MDPs.
We consider the infinite-horizon, discounted-reward MDP formulation with finite state and action spaces; we also require any trajectory with non-zero probability must have finite length (i.e., eventually reaches a terminal state).
We therefore set discount factor $\gamma=1$.


In [ ]:
from abc import ABC, abstractmethod
import numpy as np

### Infinite-horizon discounted-reward MDPs

An *infinite-horizon, discounted-reward* MDP is specified by tuple $(\mathcal{S}, \mathcal{A}, P, r, \gamma)$:
- State space $\mathcal{S}$.
- Action space $\mathcal{A}$.
- Transition function $P: \mathcal{S} \times \mathcal{A} \rightarrow \Delta(\mathcal{S})$. $P(s'|s, a)$ is the probability of transitioning into state $s'$ upon taking action $a$ in state $s$.
- Reward function $r:\mathcal{S} \times \mathcal{A} \to \mathbb{R}$. $r(s, a)$ is the immediate reward associated with taking action $a$ in state $s$.
- Discount factor $\gamma \in[0,1)$, which effectively defines a horizon.

The agent starts at some state $s_1$ drawn from initial distribution $d_1\in \Delta(\mathcal{S})$; at each timestep $h=1,2,\ldots$, the agent observes state $s_h$, picks an action $a_h$, receives reward $r_h := r(s_h, a_h)$, and then the environment transitions to a next state $s_{h+1}$ drawn from distribution $P(\cdot | s_h, a_h)$. This generates the following stochastic process of infinite length:
$$
(s_1, a_1, r_1, s_2, a_2, r_2, ...).
$$

In [ ]:
class InfiniteHorizonMDP(ABC):
    ''' Infinite-horizon, discounted-reward MDP with finite state and action spaces'''
    @abstractmethod
    def get_state_space(self) -> list:
        """ 
        Returns the state space of this MDP
        Args: None
        Returns:
            - an ordered list of all states
        """
        pass

    @property
    def nS(self) -> int:
        """ 
        Returns the number of states in this MDP
        Args: None
        Returns:
            - number of states
        """
        return len(self.get_state_space())

    @abstractmethod
    def get_action_space(self) -> list:
        """ 
        Returns the action space of this MDP
        Args: None
        Returns:
            - an ordered list of all actions
        """
        pass

    @property
    def nA(self) -> int:
        """ 
        Returns the number of actions in this MDP
        Args: None
        Returns:
            - number of actions
        """
        return len(self.get_action_space())

    @abstractmethod
    def get_discount_factor(self):
        """ 
        Returns gamma, the discount factor, of this MDP
        Args: None
        Returns:
            - gamma
        """
        pass

    @abstractmethod
    def get_reward(self, s, a):
        """ 
        Defines the (deterministic) reward function of this MDP
        Args:
            - s: state id, 0,1,...nS-1
            - a: action id, 0,1,...nA-1
        Returns:
            - reward r(s,a)
        """
        pass
    
    @abstractmethod
    def get_transition(self, s, a) -> np.ndarray:
        """ 
        Defines the transition function of this MDP
        Args:
            - s: state id, 0,1,...nS-1
            - a: action id, 0,1,...nA-1
        Returns:
            - next_s_prob:
                a np array of shape (nS,) 
                where nS is the number of states, 
                and the i-th item is the prob of transiting to i-th state 
                as ordered in method self.get_state_space
        """
        pass

    def is_terminal(self, s) -> bool:
        """ 
        Returns whether state s is a terminal state.
        s is a terminal state if, for all actions a
            - taking action a in state s leads to state s with probability 1
            - the reward for taking action a in state s is 0
        Args:
            - s: state id, 0,1,...,nS-1
        Returns:
            - True if s is a terminal state, False otherwise
        """
        for a in range(self.nA):
            next_s_prob = self.get_transition(s, a)
            if not (next_s_prob[s] == 1 and np.sum(next_s_prob) == 1):
                return False
            if self.get_reward(s, a) != 0:
                return False
        return True
    
    def get_terminal_states(self) -> list:
        """ 
        Returns a list of all terminal states in this MDP
        Args: None
        Returns:
            - a list of terminal state ids
        """
        terminal_states = []
        for s in range(self.nS):
            if self.is_terminal(s):
                terminal_states.append(s)
        return terminal_states
    
    def step(self, s, a):
        """ 
        Samples a transition from taking action a in state s,
        i.e., the generative model of this MDP
        Args:
            - s: state id,  0,1,...,nS-1
            - a: action id, 0,1,...,nA-1
        Returns:
            - a sampled next state: a dict of following keys
            - reward r(s,a) per reward function
        """

        next_s_prob = self.get_transition(s, a)
        next_s = np.random.choice(self.nS, p=next_s_prob)
        r = self.get_reward(s, a)
        return next_s, r

### Example: Combination Lock as a infinite-horizon MDP

Consider the MDP depicted below, with $H+2$ states, and two actions $a_g$ and $a_b$, and a starting state 1 . The "good" action $a_g$ deterministically leads to the next state in the chain, while the "bad" action deterministically leads to a terminal state. The only place where a non-zero reward can be received is the last state $H$, if the good action is chosen.

<img src="combination_lock.png" alt="combination_lock" width="700"/>

In [ ]:
class CombinationLock(InfiniteHorizonMDP):
    """ 
    Combination lock as an infinite-horizon MDP with no discounting (gamma=1)
    """
    def __init__(self, H):
        self.H = H
        self.state_space = list(range(H + 2)) # states: 1 to H+1 are the chain states; 0 is a terminal state
        self.action_space = [0, 1]  # actions: 0 (good), 1 (bad)
        self.gamma = 1.0           # no discounting

    def get_state_space(self):
        return self.state_space
    
    def get_action_space(self):
        return self.action_space
    
    def get_discount_factor(self):
        return self.gamma

    def get_reward(self, s, a):
        H = self.H
        if s == 0 or s == H+1:  # terminal state
            return 0.
        elif s == H and a == 0: # good action end of chain
            return H + 10.      
        elif a == 0:            # good action mid of chain
            return -1.
        else:                   # bad action mid of chain
            return 1.           
    
    def get_transition(self, s, a):
        nS = len(self.state_space) # H + 2
        next_s_prob = np.zeros(nS)
        if s == 0 or s == self.H + 1: # terminal states
            next_s_prob[s] = 1.0
        elif 1 <= s <= self.H:
            if a == 0:
                next_s_prob[s + 1] = 1.0
            else:
                next_s_prob[0] = 1.0
        return next_s_prob

### Tree Node

We first implement the tree node.
We here take the approach that distinguishes between *VNode* vs *QNode*
- VNode: a tree node that is in some state but has not committed to any action 
- QNode: a tree node that has committed to some action in a state; the state is in its parent that is a VNode

The two types of nodes are interleaved: a VNode has only QNodes as its children, and a QNode has only VNodes (next states) as its children.

In [ ]:
class VNode:
    def __init__(self, state, parent=None):
        self.state = state
        self.parent = parent # Parent QNode
        self.children = {} # Maps actions to child QNodes
    
    @property
    def visit_count(self):
        count = 0
        for qnode in self.children.values():
            count += qnode.visit_count
        return count

class QNode:
    def __init__(self, parent, action, reward):
        self.parent = parent # Parent VNode
        self.action = action
        self.reward = reward
        self.children = {}   # Maps states to child VNodes
        self.visit_count = 0
        self.total_value = 0.0

    def get_average_value(self):
        if self.visit_count == 0:
            return 0.0
        return self.total_value / self.visit_count

In [ ]:
class MCTS(ABC):
    def __init__(self, mdp):
        # We only need the generative model of the MDP
        self.terminal_states = mdp.get_terminal_states()
        self.nS = mdp.nS
        self.nA = mdp.nA
        self.mdp_step = mdp.step # generative model, i.e., the function to sample (s',r) given (s,a)
    
    @abstractmethod
    def select_action(self, vnode):
        """select an action from the given vnode"""
        pass
    
    def tree_policy(self, vnode):
        """
        Traverses the tree from the given vnode
        until it reaches an unobserved transition (s,a,s').
        Expands the tree by adding the new transition.
        Return:
            - the new vnode in s'        
        """
        while vnode.state not in self.terminal_states:
            # Select an action to take in the current vnode
            action = self.select_action(vnode)
            # Sample transition
            next_state, reward = self.mdp_step(vnode.state, action)

            # Check if (s,a,s') is already in the tree
            if action in vnode.children and next_state in vnode.children[action].children:
                # (s,a,s') is already in the tree, so move to the next vnode
                vnode = vnode.children[action].children[next_state]
            else:
                # (s,a,s') is not in the tree, so expand the tree
                if action not in vnode.children:
                    vnode.children[action] = QNode(vnode, action, reward)
                qnode = vnode.children[action]
                if next_state not in qnode.children:
                    qnode.children[next_state] = VNode(next_state, qnode)
                vnode = qnode.children[next_state]
                return vnode
        
        # Reached a terminal state
        return vnode
  
    @abstractmethod
    def default_policy(self, state):
        """
        Default policy for MCTS.
        Usually a random rollout policy that samples actions uniformly at random
        until it reaches a terminal state.

        Returns
            - total reward of the trajectory ending in a terminal state
        """
        pass
    
    def backup(self, vnode, total_reward):
        """
        Backpropagates the total reward from the given vnode to its ancestors.
        """
        qnode = vnode.parent
        while qnode is not None:
            qnode.visit_count += 1
            # update total_reward to include reward at this qnode
            total_reward = qnode.reward + total_reward
            qnode.total_value += total_reward
            # move up to the parent qnode
            qnode = qnode.parent.parent
    
    def best_action(self, vnode):
        """
        Returns the action with the highest average value from the given vnode.
        """
        best_action = np.random.choice(self.nA) # Default action as random tie-break
        best_value = float('-inf')
        for action, qnode in vnode.children.items():
            avg_value = qnode.get_average_value()
            if avg_value > best_value:
                best_value = avg_value
                best_action = action
        return best_action
    
    def search(self, root: VNode, n_simulations):
        """
        Performs MCTS starting from the given root vnode for a specified number of simulations.
        After the simulations, selects the best action from the root vnode.

        Args:
            - root: the root VNode to start the search from
            - n_simulations: number of MCTS simulations to perform

        Returns:
            - the best action from the root vnode after MCTS
        """
        for _ in range(n_simulations):
            # 1. Tree policy: traverse the tree to find a leaf vnode
            leaf_vnode = self.tree_policy(root)
            # 2. Default policy: simulate a random rollout from the leaf vnode
            total_reward = self.default_policy(leaf_vnode.state)
            # 3. Backup: backpropagate the reward to update the tree
            self.backup(leaf_vnode, total_reward)
        
        # 4. Select the best action from the root vnode
        return self.best_action(root)

In [ ]:
class UCT(MCTS):
    def __init__(self, mdp, c=1.0):
        super().__init__(mdp)
        self.c = c

    def select_action(self, vnode):
        """select an action from the given vnode using UCB"""
        # explore unvisited action
        for action in range(self.nA):
            if action not in vnode.children:
                return action
        # If all actions explored, select the one with highest UCB
        best_action = np.random.choice(self.nA)
        best_value = float('-inf')
        for action, qnode in vnode.children.items():
            ucb_value = qnode.get_average_value() + self.c * np.sqrt(np.log(vnode.visit_count) / (qnode.visit_count + 1e-6))
            if ucb_value > best_value:
                best_value = ucb_value
                best_action = action
        return best_action

    def default_policy(self, state):
        """
        Starting from state, perform a random rollout 
        that samples actions uniformly at random
        until it reaches a terminal state.

        Returns
            - total reward of the trajectory ending in a terminal state
        """
        total_reward = 0.0
        while state not in self.terminal_states:
            action = np.random.choice(self.nA)
            next_state, reward = self.mdp_step(state, action)
            total_reward +=  reward
            state = next_state
        return total_reward

### Run MCTS on Combination Lock

In [ ]:
def run_episode_by_mcts(mdp, initial_state, mcts, n_simulations, max_steps=100):
    """
    Runs a single episode in the given MDP using MCTS for action selection.

    Args:
        - mdp: the MDP to run the episode in
        - mcts: the MCTS instance to use for action selection
        - initial_state: the starting state of the episode
        - n_simulations: number of MCTS simulations per step
        - max_steps: maximum number of steps to run in the episode

    Returns:
        - total_reward: total reward accumulated during the episode
    """
    state = initial_state
    total_reward = 0.0
    steps = 0

    root = VNode(state)

    while state not in mdp.get_terminal_states() and steps < max_steps:
        action = mcts.search(root, n_simulations)
        next_state, reward = mdp.step(state, action)

        total_reward += reward
        steps += 1

        # Update the tree with the new state
        if action not in root.children:
            root.children[action] = QNode(root, action, reward)
        qnode = root.children[action]
        if next_state not in qnode.children:
            qnode.children[next_state] = VNode(next_state, qnode)
        root = qnode.children[next_state]
        state = next_state

    return total_reward


In [ ]:
# Run MCTS on Combination Lock with varying number of simulations and exploration constants (c)
mdp = CombinationLock(H=5)
initial_state = 1
c_list = [1.0, 2.0, 5.0, 10.0]
n_simulations_list = [10, 100, 1000, 10000]

for c in c_list:
    print(f"Exploration constant c = {c}")
    mcts = UCT(mdp, c=c)
    for n_simulations in n_simulations_list:
        total_reward = run_episode_by_mcts(mdp, initial_state, mcts, n_simulations, max_steps=100)
        print(f"  #Simulations per step: {n_simulations}, Total Reward: {total_reward}")
    print()
